# **Data Cleaning**

## Purpose
This notebook applies the cleaning decisions identified in `01_Auditing_Data.ipynb`.

The workflow is:

1. Load the raw dataset.
2. Preserve the original data by working on a separate dataframe.
3. Confirm duplicate status.
4. Handle selected missing values.
5. Standardize product categories.
6. Check and clean whitespace in key categorical fields.
7. Validate data types and relationships.
8. Perform final quality checks.
9. Save and verify the cleaned dataset.

**Important:** EDA and predictive modeling are not performed in this notebook.


# **1. Environment Setup**

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import pandas as pd

In [3]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 50)

project_root = "/content/drive/MyDrive/Consumer Complain Project"
raw_file_path = os.path.join(
    project_root,
    "data/raw/complaints_100k.parquet"
)
processed_dir = os.path.join(project_root, "data/processed")
output_path = os.path.join(
    processed_dir,
    "complaints_cleaned.parquet"
)

os.makedirs(processed_dir, exist_ok=True)

# **2. Load Raw Data and Create a Working Copy**

In [4]:
raw_df = pd.read_parquet(raw_file_path)

# Work on a copy so the raw dataframe remains unchanged.
df = raw_df.copy()

print("Raw dataset loaded successfully.")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]:,}")

Raw dataset loaded successfully.
Rows    : 90,112
Columns : 16


In [5]:
df.head()

,date_received,product,sub_product,issue,sub_issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response,complaint_id
0,2026-04-25 20:09:10+00:00,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Investigation took more than 30 days,None,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",WA,98087,None,Web,2026-04-25 20:17:54+00:00,Closed with explanation,Yes,21603527
1,2026-02-14 21:18:34+00:00,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Account information incorrect,None,Company has responded to the consumer and the ...,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AL,35022,None,Web,2026-02-14 21:19:00+00:00,Closed with explanation,Yes,19508852
2,2026-05-25 18:41:22+00:00,Credit reporting or other personal consumer re...,Credit reporting,Problem with a company's investigation into an...,Their investigation did not fix an error on yo...,None,None,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",TX,75287,None,Web,2026-05-25 18:45:42+00:00,None,Yes,22543000
3,2026-05-12 15:06:42+00:00,Debt collection,I do not know,Communication tactics,"You told them to stop contacting you, but they...",None,None,Western Management Consultants,NY,146XX,None,Web,2026-05-12 15:12:25+00:00,Closed with explanation,Yes,22116348
4,2026-05-01 08:25:14+00:00,Credit reporting or other personal consumer re...,Credit reporting,Improper use of your report,Reporting company used your report improperly,None,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,SC,29455,None,Web,2026-05-01 08:25:37+00:00,Closed with explanation,Yes,21783248


# **3. Initial Data Quality Check**

In [6]:
print("Initial Quality Check")
print("---------------------")
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate complaint IDs:", df["complaint_id"].duplicated().sum())

missing_before = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

missing_before = (
    missing_before[missing_before["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

print("\nMissing values before cleaning:")
display(missing_before)

Initial Quality Check
---------------------
Duplicate rows: 0
Duplicate complaint IDs: 0

Missing values before cleaning:


,missing_count,missing_percentage
tags,85398,94.77
consumer_complaint_narrative,70000,77.68
company_public_response,54692,60.69
company_response_to_consumer,15807,17.54
sub_issue,4017,4.46
date_sent_to_company,3180,3.53
state,230,0.26
zip_code,170,0.19


### Cleaning decision for duplicates
The audit found no duplicate rows and no duplicate `complaint_id` values. Therefore, no records are removed in this step. The duplicate checks are retained as evidence of data integrity.

# **4. Blank and Whitespace Handling**

Before standardizing categories, check for empty strings and leading/trailing whitespace. Whitespace-only values are converted to missing values only in fields where blank text is detected. Key categorical columns are then stripped of leading and trailing spaces.


In [7]:
key_text_columns = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "company",
    "state",
    "tags",
    "submitted_via",
    "timely_response"
]

blank_before = {}

for column in key_text_columns:
    blank_before[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

print("Blank or Whitespace-Only Values Before Cleaning")
print("------------------------------------------------")
display(pd.Series(blank_before, name="blank_count").to_frame())

Blank or Whitespace-Only Values Before Cleaning
------------------------------------------------


,blank_count
product,0
sub_product,0
issue,0
sub_issue,4017
company,0
state,230
tags,85398
submitted_via,0
timely_response,0


In [8]:
# Standardize leading and trailing whitespace in key categorical columns.
for column in key_text_columns:
    if df[column].dtype == "object" or str(df[column].dtype).startswith("string"):
        df[column] = df[column].where(
            df[column].isna(),
            df[column].astype(str).str.strip()
        )

# Convert any remaining whitespace-only strings in key fields to missing values.
for column in key_text_columns:
    df[column] = df[column].replace("", pd.NA)

print("Whitespace standardization completed.")

Whitespace standardization completed.


In [9]:
blank_after = {}

for column in key_text_columns:
    blank_after[column] = (
        df[column]
        .fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

print("Blank or Whitespace-Only Values After Cleaning")
print("------------------------------------------------")
display(pd.Series(blank_after, name="blank_count").to_frame())

Blank or Whitespace-Only Values After Cleaning
------------------------------------------------


,blank_count
product,0
sub_product,0
issue,0
sub_issue,4017
company,0
state,230
tags,85398
submitted_via,0
timely_response,0


# **5. Missing Value Treatment**

### 5.1 Fields where missing values represent an unknown or absent category

The following categorical fields are filled with explicit labels so that missingness is retained as information instead of silently dropping records:

- `sub_issue` → `Not provided`
- `state` → `Unknown`
- `zip_code` → `Unknown`
- `tags` → `No tag`

In [10]:
missing_fill_map = {
    "sub_issue": "Not provided",
    "state": "Unknown",
    "zip_code": "Unknown",
    "tags": "No tag"
}

for column, fill_value in missing_fill_map.items():
    df[column] = df[column].fillna(fill_value)

print("Selected categorical missing values handled successfully.")

Selected categorical missing values handled successfully.


### 5.2 Fields intentionally left missing

The following columns are **not imputed** because replacing missing values with invented values could change their meaning:

- `consumer_complaint_narrative`
- `company_public_response`
- `company_response_to_consumer`
- `date_sent_to_company`

These columns retain their original missing values. Their treatment depends on the purpose of later analysis or modeling.

In [11]:
intentional_missing_columns = [
    "consumer_complaint_narrative",
    "company_public_response",
    "company_response_to_consumer",
    "date_sent_to_company"
]

missing_after_selected_cleaning = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

print("Remaining Missing Values")
print("------------------------")
display(
    missing_after_selected_cleaning[
        missing_after_selected_cleaning["missing_count"] > 0
    ].sort_values("missing_count", ascending=False)
)

Remaining Missing Values
------------------------


,missing_count,missing_percentage
consumer_complaint_narrative,70000,77.68
company_public_response,54692,60.69
company_response_to_consumer,15807,17.54
date_sent_to_company,3180,3.53


# **6. Product Category Standardization**

The audit identified legacy or inconsistent product labels. They are standardized to the project's current product categories.

The `Credit card or prepaid card` category is split using its sub-product:
- General-purpose credit card or charge card → `Credit card`
- General-purpose prepaid card → `Prepaid card`

In [12]:
print("Rows Affected by Product Standardization (Before Change)")
print("----------------------------------------------------------")

rare_products = [
    "Credit reporting, credit repair services, or other personal consumer reports",
    "Consumer Loan",
    "Bank account or service",
    "Credit card or prepaid card"
]

display(df[df["product"].isin(rare_products)][
    ["product", "sub_product", "issue", "complaint_id"]
])

Rows Affected by Product Standardization (Before Change)
----------------------------------------------------------


,product,sub_product,issue,complaint_id
2836,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,7372634
3570,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,7377762
6193,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,7333668
7391,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,7333960
8646,"Credit reporting, credit repair services, or o...",Credit reporting,Problem with a credit reporting company's inve...,7402157
8968,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,7370341
14757,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,7271049
15132,"Credit reporting, credit repair services, or o...",Other personal consumer report,Incorrect information on your report,5581418
16968,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,5685534
21746,"Credit reporting, credit repair services, or o...",Credit reporting,Improper use of your report,6713420


In [13]:
product_standardization = {
    "Credit reporting, credit repair services, or other personal consumer reports":
        "Credit reporting or other personal consumer reports",
    "Consumer Loan":
        "Payday loan, title loan, personal loan, or advance loan",
    "Bank account or service":
        "Checking or savings account"
}

df["product"] = df["product"].replace(product_standardization)

df.loc[
    (df["product"] == "Credit card or prepaid card")
    & (df["sub_product"] == "General-purpose credit card or charge card"),
    "product"
] = "Credit card"

df.loc[
    (df["product"] == "Credit card or prepaid card")
    & (df["sub_product"] == "General-purpose prepaid card"),
    "product"
] = "Prepaid card"

print("Product category standardization completed.")

Product category standardization completed.


In [14]:
print("Product Categories After Standardization")
print("----------------------------------------")
print(df["product"].value_counts(dropna=False))

Product Categories After Standardization
----------------------------------------
product
Credit reporting or other personal consumer reports        63131
Debt collection                                             9700
Checking or savings account                                 4264
Credit card                                                 4220
Money transfer, virtual currency, or money service          2034
Mortgage                                                    1758
Vehicle loan or lease                                       1444
Payday loan, title loan, personal loan, or advance loan     1122
Student loan                                                1011
Prepaid card                                                 728
Debt or credit management                                    655
Other                                                         45
Name: count, dtype: int64


# **7. Data Type Validation**

In [15]:
print("Key Data Types")
print("--------------")

key_columns = [
    "date_received",
    "date_sent_to_company",
    "zip_code",
    "complaint_id",
    "timely_response"
]

for column in key_columns:
    print(f"{column}: {df[column].dtype}")

Key Data Types
--------------
date_received: datetime64[us, UTC]
date_sent_to_company: datetime64[us, UTC]
zip_code: string
complaint_id: string
timely_response: object


# **8. Relationship and Consistency Validation**

In [16]:
print("Product and Sub-Product Coverage")
print("--------------------------------")

subproducts_per_product = (
    df.groupby("product")["sub_product"]
      .nunique(dropna=True)
      .sort_values(ascending=False)
)

display(subproducts_per_product.to_frame("unique_sub_products"))

Product and Sub-Product Coverage
--------------------------------


,unique_sub_products
product,
Debt collection,12
Mortgage,9
"Payday loan, title loan, personal loan, or advance loan",8
"Money transfer, virtual currency, or money service",7
Prepaid card,5
Debt or credit management,4
Checking or savings account,4
Credit card,2
Credit reporting or other personal consumer reports,2


In [17]:
print("Product and Sub-Product Combinations")
print("------------------------------------")

product_subproduct = (
    df[["product", "sub_product"]]
    .drop_duplicates()
    .sort_values(["product", "sub_product"])
)

display(product_subproduct)

Product and Sub-Product Combinations
------------------------------------


,product,sub_product
59,Checking or savings account,CD (Certificate of Deposit)
134,Checking or savings account,Checking account
55,Checking or savings account,Other banking product or service
608,Checking or savings account,Savings account
8,Credit card,General-purpose credit card or charge card
315,Credit card,Store credit card
0,Credit reporting or other personal consumer re...,Credit reporting
2127,Credit reporting or other personal consumer re...,Other personal consumer report
417,Debt collection,Auto debt
27135,Debt collection,Credit card


# **9. Outlier Handling Decision**

This dataset is primarily categorical, text-based, and date-based. At this cleaning stage, there is no original continuous measurement selected for automatic outlier removal.

No rows are removed as outliers because an extreme value is not automatically an invalid value. Any derived continuous variables created later (for example, response-time duration) should be analyzed in EDA and handled according to the analytical objective.

This avoids deleting valid complaint records solely because a derived value is extreme.


# **10. Final Data Quality Verification**

In [18]:
final_quality_summary = {
    "rows": len(df),
    "columns": len(df.columns),
    "duplicate_rows": df.duplicated().sum(),
    "duplicate_complaint_ids": df["complaint_id"].duplicated().sum(),
    "missing_sub_issue": df["sub_issue"].isna().sum(),
    "missing_state": df["state"].isna().sum(),
    "missing_zip_code": df["zip_code"].isna().sum(),
    "missing_tags": df["tags"].isna().sum()
}

print("Final Quality Verification")
print("--------------------------")

for metric, value in final_quality_summary.items():
    print(f"{metric}: {value:,}")

Final Quality Verification
--------------------------
rows: 90,112
columns: 16
duplicate_rows: 0
duplicate_complaint_ids: 0
missing_sub_issue: 0
missing_state: 0
missing_zip_code: 0
missing_tags: 0


In [19]:
if final_quality_summary["duplicate_complaint_ids"] == 0:
    print("\nPASS: All complaint IDs are unique.")
else:
    print("\nFAIL: Duplicate complaint IDs found — review before saving.")


PASS: All complaint IDs are unique.


In [20]:
remaining_missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
})

remaining_missing = (
    remaining_missing[remaining_missing["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

print("Remaining Missing Values (Intentionally Retained)")
print("-------------------------------------------------")
display(remaining_missing)

Remaining Missing Values (Intentionally Retained)
-------------------------------------------------


,missing_count,missing_percentage
consumer_complaint_narrative,70000,77.68
company_public_response,54692,60.69
company_response_to_consumer,15807,17.54
date_sent_to_company,3180,3.53


# **11. Save the Cleaned Dataset**

In [21]:
df.to_parquet(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("File:", output_path)

Cleaned dataset saved successfully.
File: /content/drive/MyDrive/Consumer Complain Project/data/processed/complaints_cleaned.parquet


# **12. Reload and Verify the Saved Dataset**

In [22]:
cleaned_df = pd.read_parquet(output_path)

print("Saved Dataset Verification")
print("-------------------------")
print(f"Rows    : {cleaned_df.shape[0]:,}")
print(f"Columns : {cleaned_df.shape[1]:,}")
print("Duplicate rows:", cleaned_df.duplicated().sum())
print("Duplicate complaint IDs:", cleaned_df["complaint_id"].duplicated().sum())

Saved Dataset Verification
-------------------------
Rows    : 90,112
Columns : 16
Duplicate rows: 0
Duplicate complaint IDs: 0


In [23]:
print("Verification of Cleaned Fields")
print("------------------------------")

verification_columns = [
    "sub_issue",
    "state",
    "zip_code",
    "tags"
]

for column in verification_columns:
    print(f"Missing {column}: {cleaned_df[column].isna().sum():,}")

Verification of Cleaned Fields
------------------------------
Missing sub_issue: 0
Missing state: 0
Missing zip_code: 0
Missing tags: 0


# **13. Cleaning Summary and Handoff**

### Completed cleaning actions
- Preserved the raw dataframe and worked on a copy.
- Verified that no duplicate rows or duplicate complaint IDs existed.
- Filled selected categorical missing values with explicit labels.
- Retained high-missingness and context-dependent fields without inventing values.
- Standardized leading and trailing whitespace in key categorical fields.
- Standardized identified legacy/inconsistent product categories.
- Validated data types and product/sub-product relationships.
- Made and documented the outlier-handling decision.
- Saved the cleaned dataset as `complaints_cleaned.parquet`.
- Reloaded the saved file and verified its integrity.

### Next notebook
Proceed to `03_EDA.ipynb`, which should load only:

`data/processed/complaints_cleaned.parquet`

EDA should not repeat the cleaning process. It should focus on understanding complaint patterns, distributions, relationships, and key findings.
